# 1. Data preparation

**DATA3800, autumn 2026. Group 8. Forecasting hourly electricity consumption in Oslo (NO1).**

This notebook goes from the two raw files to the dataset the models use:

1. Load the raw data
2. Check the quality of the raw data
3. Clean and merge
4. Build the features
5. Save `data/processed/dataset.csv`

The functions live in `src/build_dataset.py` and `src/features.py`, so the same code runs here, from the command line and inside the agent.

**Data sources.** Hourly actual load and ENTSO-E's official day-ahead load forecast for bidding zone NO1 come from the ENTSO-E Transparency Platform. Hourly air temperature at Oslo Blindern (station SN18700) comes from the Frost API at met.no. The raw files are included in `data/raw/`, so this notebook runs without API keys. To download them again, put your own keys in `.env` and run `python src/fetch_consumption.py` and `python src/fetch_weather.py`.

In [1]:
# Setup: find the repo, make src/ importable, and print the environment.
import platform, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib, numpy as np, pandas as pd, seaborn, sklearn
from IPython.display import Image, display

print("Python      ", platform.python_version())
for lib in (pd, np, sklearn, matplotlib, seaborn):
    print(f"{lib.__name__:<12}", lib.__version__)

pd.set_option("display.width", 160)

def show(name, width=820):
    """Display a figure that a pipeline step saved in figures/."""
    display(Image(filename=str(ROOT / "figures" / f"{name}.png"), width=width))

Python       3.14.2
pandas       3.0.6
numpy        2.5.4
sklearn      1.9.1
matplotlib   3.11.2
seaborn      0.13.2


## 1.1 Load the raw data

One row is one hour. All timestamps are stored in UTC and converted to Oslo time only when we build calendar features and plots.

In [2]:
import config
import build_dataset

load, temp = build_dataset.load_raw()
print("Consumption:", load.shape, "| Temperature:", temp.shape)
display(load.head(3))
display(temp.head(3))

Consumption: (32855, 2) | Temperature: (32856, 1)


,load_mw,forecast_mw
time_utc,,
2022-12-31 23:00:00+00:00,4444.18,4383.78
2023-01-01 00:00:00+00:00,4423.47,4236.76
2023-01-01 01:00:00+00:00,4414.98,4332.66


,temp_c
time_utc,
2023-01-01 00:00:00+00:00,-1.95
2023-01-01 01:00:00+00:00,-2.25
2023-01-01 02:00:00+00:00,-2.45


| Column | Meaning | Type |
| --- | --- | --- |
| `load_mw` | Actual total load in NO1, hourly mean (the target) | numerical |
| `forecast_mw` | ENTSO-E's day-ahead forecast of the same hour (benchmark only, never a feature) | numerical |
| `temp_c` | Measured air temperature at Oslo Blindern | numerical |

## 1.2 Quality check

An hourly series has exactly one row per hour. That gives us several numbers we know in advance: the number of rows between the first and last hour, zero duplicate timestamps, and 8760 hours in a normal year (8784 in a leap year). Checking against known answers finds problems that scrolling through the file would not.

In [3]:
report = build_dataset.quality_report(load, temp)

Data quality check on the raw files:
 file                 consumption_no1.csv temperature_blindern.csv
first_hour_utc          2022-12-31 23:00         2023-01-01 00:00
last_hour_utc           2026-09-30 21:00         2026-09-30 23:00
rows                               32855                    32856
expected_hours                     32855                    32856
duplicate_timestamps                   0                        0
missing_hours                          0                        0
longest_gap_hours                      0                        0
values_outside_range                   0                        0
min                               1759.5                    -22.5
max                               8166.0                     32.4
Hours with load per year (a full year has 8760, a leap year 8784):
 time_utc
2023    8760
2024    8784
2025    8760
2026    6551


Both files are complete: no duplicate timestamps, no missing hours and no values outside a plausible range. 2024 has 8784 hours because it is a leap year, and 2026 stops on 30 September. So the cleaning step below has very little to repair, and we say so instead of inventing a cleaning problem.

## 1.3 Clean and merge

The two files are joined on the timestamp. Gaps of up to three hours would be filled by linear interpolation; longer gaps would be left empty and dropped. The only empty values here are at the two ends of the period, where one file starts or stops an hour before the other.

In [4]:
df = load.join(temp, how="outer")
df = build_dataset.clean(df)

Missing values before filling:
 load_mw        2
forecast_mw    2
temp_c         1
dtype: int64
Missing values after filling:
 load_mw        2
forecast_mw    2
temp_c         1
dtype: int64


## 1.4 Features

| Feature | Why |
| --- | --- |
| `hour`, `weekday`, `month` | Daily, weekly and yearly rhythm, in Oslo time |
| `is_weekend`, `is_holiday` | Days when offices and industry are closed (Norwegian public holidays) |
| `load_lag_48h` | Load two days earlier. A day-ahead forecast is made the day before, so yesterday's full load is not known yet. 48 hours is the newest value we are sure to have |
| `load_lag_168h` | Load at the same hour one week earlier |
| `temp_c` | Temperature. Electric heating makes this the main driver of load in Norway |

`day_type`, `lag48_same_daytype` and `load_lag_daytype` belong to the weekend-fix experiment in notebook 3.

In [5]:
df = build_dataset.add_features(df)
before = len(df)
df = df.dropna()
print(f"Dropped {before - len(df)} rows without a full week of history (the first 168 hours) or at the edges.")
print(f"Final dataset: {len(df):,} rows, {df.shape[1]} columns, {df.index.min():%Y-%m-%d} to {df.index.max():%Y-%m-%d}")
df.head(3)

Dropped 170 rows without a full week of history (the first 168 hours) or at the edges.
Final dataset: 32,687 rows, 13 columns, 2023-01-07 to 2026-09-30


,load_mw,forecast_mw,temp_c,hour,weekday,month,is_weekend,is_holiday,load_lag_48h,load_lag_168h,day_type,lag48_same_daytype,load_lag_daytype
time_utc,,,,,,,,,,,,,
2023-01-07 23:00:00+00:00,4579.48,4578.33,2.35,0,6,1,1,0,5317.97,4444.18,1,0,4444.18
2023-01-08 00:00:00+00:00,4414.00,4500.82,3.50,1,6,1,1,0,5222.65,4423.47,1,0,4423.47
2023-01-08 01:00:00+00:00,4264.25,4372.36,4.25,2,6,1,1,0,5159.61,4414.98,1,0,4414.98


In [6]:
df.describe().T[["count", "mean", "std", "min", "max"]].round(1)

,count,mean,std,min,max
load_mw,32687.0,3924.6,1313.2,1759.5,8166.0
forecast_mw,32687.0,3951.7,1320.2,1712.2,8187.3
temp_c,32687.0,8.0,8.6,-22.5,32.4
hour,32687.0,11.5,6.9,0.0,23.0
weekday,32687.0,3.0,2.0,0.0,6.0
month,32687.0,6.2,3.3,1.0,12.0
is_weekend,32687.0,0.3,0.5,0.0,1.0
is_holiday,32687.0,0.0,0.2,0.0,1.0
load_lag_48h,32687.0,3927.9,1314.5,1759.5,8166.0
load_lag_168h,32687.0,3937.6,1318.6,1759.5,8166.0


## 1.5 Save

In [7]:
config.PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(config.PROCESSED / "dataset.csv")
print("Saved", (config.PROCESSED / "dataset.csv").relative_to(config.ROOT))

Saved data/processed/dataset.csv
